# Selection Efficiency audit (Colab)

**Runtime → Change runtime type → T4 GPU.** Then run all cells.

Everything is written to your Google Drive (`MyDrive/se_audit_run`) after every batch, so **if Colab disconnects, nothing is lost**: reconnect, choose Runtime → Run all again, and it continues from the last checkpoint. Finished splits are skipped without reloading the model.

Run only one Colab session at a time on this folder. A second session is refused while the first is alive.

Expected total on one T4: about 2–2.5 h of GPU time, which can be spread over several sessions.

In [ ]:
REPO = 'https://github.com/manyagupta13/se-audit.git'
WORK = '/content/drive/MyDrive/se_audit_run'   # checkpoints live here
MODE = 'full'                                   # 'full', 'smoke', or 'status'

In [ ]:
# Mount Drive and get the latest code
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/se_audit/.git'):
    !git -C /content/se_audit pull -q
else:
    !git clone -q {REPO} /content/se_audit
!git -C /content/se_audit log -1 --format='code version: %h  %s'
!pip install -q 'sentence-transformers>=2.6' scipy 2>&1 | tail -1
!nvidia-smi -L

In [ ]:
# Where things stand (safe to run any time)
!python /content/se_audit/run_pipeline.py --work {WORK} --status

In [ ]:
# Run (resumes automatically). Progress is printed every 5 minutes.
!python /content/se_audit/run_pipeline.py --work {WORK} --{MODE}

In [ ]:
# Download the results (also kept in Drive at WORK)
from google.colab import files
for f in ['tables.zip', 'raw_outputs.zip']:
    p = os.path.join(WORK, f)
    print(p, 'exists' if os.path.exists(p) else 'not yet')
    if os.path.exists(p): files.download(p)

**If the run cell is refused with “Another session … may still be running”:** the previous runtime wrote to the folder in the last 2–3 minutes. Wait that long and rerun, or, if you are sure the old session is gone, add `--force` to the run command.